### Importing the required libraries

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
import joblib as jl

### Loading the excel dataset into pandas

In [9]:
mdf = pd.read_excel("/Users/samhithkonidena/My Data Journey/Data Job Prep/Portfolio/Analysis/Titanic_Analysis/Cleaned_Data/Titanic_Cleaned_Data.xlsx")
mdf.head()

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,cabin_known,age_category,ship_class,survivor?
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0,Youth,Third Class,No
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1,Adult,First Class,Yes
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0,Adult,Third Class,Yes
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1,Adult,First Class,Yes
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0,Adult,Third Class,No


#### Deriving the `family_size` and `is_alone` features/columns

In [10]:
mdf["family_size"] = mdf["sibsp"] + mdf["parch"] + 1
mdf["is_alone"] = mdf["family_size"] == 1
mdf.head()

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,cabin_known,age_category,ship_class,survivor?,family_size,is_alone
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0,Youth,Third Class,No,2,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1,Adult,First Class,Yes,2,False
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0,Adult,Third Class,Yes,1,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1,Adult,First Class,Yes,2,False
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0,Adult,Third Class,No,1,True


### Defining the 'X' and 'Y' feature datasets for the model to train on

In [11]:
X = mdf[["pclass", "sex", "age", "embarked", "family_size", "is_alone"]]
y = mdf["survived"]

X.head()
X.shape
y.head()
y.shape

(891,)

### Splitting both datasets for the model to train and test on 

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Checking the target dataset distribution

In [13]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)
X.head()

,pclass,sex,age,embarked,family_size,is_alone
0,3,male,22.0,S,2,False
1,1,female,38.0,C,2,False
2,3,female,26.0,S,1,True
3,1,female,35.0,S,2,False
4,3,male,35.0,S,1,True


### Creating feature lists for encoding/preprocessing

In [14]:
num_features = ["pclass", "age", "family_size", "is_alone"]
cat_features = ["sex", "embarked"]

### Creating the two column transformers for processing two kinds of features

In [15]:
num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(handle_unknown='ignore')

### Applying the transformers on the features

In [16]:
preprocessor = ColumnTransformer([
    ("numbers", num_transformer, num_features),
    ("categories", cat_transformer, cat_features)
])

### Introducing/defining the model

In [17]:
model = LogisticRegression()

### Creating the model processing pipeline

In [18]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

### Training the model

In [19]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['pclass','sex','age','embarked','family_size','is_alone']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numbers', ...), ('categories', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rema

#### Generating the `X_test` prediction and probability

In [20]:
y_pred = pipeline.predict(X_test)
print(y_pred)

[0 0 0 0 1 1 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0
 0 0 1 0 1 0 1 1 0 0 1 1 1 1 1 1 0 1 0 0 0 1 0 1 1 0 0 1 1 1 0 0 0 1 0 1 1
 0 0 0 0 0 1 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 1 1 0 0 0 0 1 1 1 0 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 1 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0
 1 1 0 0 1 1 0 1 0 0 0 1 1 1 1 1 1 1 0 0 1 1 0 0 1 0 1 0 0 1 0]


In [21]:
y_pred_proba = pipeline.predict_proba(X_test)
print(y_pred_proba)

[[0.90239652 0.09760348]
 [0.93486827 0.06513173]
 [0.85993144 0.14006856]
 [0.9451568  0.0548432 ]
 [0.23656938 0.76343062]
 [0.44878793 0.55121207]
 [0.26880547 0.73119453]
 [0.68801535 0.31198465]
 [0.63360728 0.36639272]
 [0.76549316 0.23450684]
 [0.84870179 0.15129821]
 [0.87337009 0.12662991]
 [0.4462763  0.5537237 ]
 [0.7316492  0.2683508 ]
 [0.66596941 0.33403059]
 [0.8340437  0.1659563 ]
 [0.59380035 0.40619965]
 [0.91663008 0.08336992]
 [0.8802155  0.1197845 ]
 [0.25465184 0.74534816]
 [0.91663008 0.08336992]
 [0.19303853 0.80696147]
 [0.92206058 0.07793942]
 [0.51264034 0.48735966]
 [0.91938704 0.08061296]
 [0.05127962 0.94872038]
 [0.84870179 0.15129821]
 [0.64134989 0.35865011]
 [0.88806629 0.11193371]
 [0.88437289 0.11562711]
 [0.93196063 0.06803937]
 [0.08535195 0.91464805]
 [0.88160919 0.11839081]
 [0.90783838 0.09216162]
 [0.84219769 0.15780231]
 [0.81630561 0.18369439]
 [0.84219769 0.15780231]
 [0.69390526 0.30609474]
 [0.84870179 0.15129821]
 [0.15619647 0.84380353]


In [22]:
y_pred[:10]

array([0, 0, 0, 0, 1, 1, 1, 0, 0, 0])

In [23]:
y_pred_proba[:10]

array([[0.90239652, 0.09760348],
       [0.93486827, 0.06513173],
       [0.85993144, 0.14006856],
       [0.9451568 , 0.0548432 ],
       [0.23656938, 0.76343062],
       [0.44878793, 0.55121207],
       [0.26880547, 0.73119453],
       [0.68801535, 0.31198465],
       [0.63360728, 0.36639272],
       [0.76549316, 0.23450684]])

### Calculating model accuracy

In [24]:
accuracy = accuracy_score(y_test, y_pred)
print (accuracy*100)

79.3296089385475


### Calculating model confusion

In [25]:
confusion = confusion_matrix(y_test, y_pred)
print(confusion)

[[94 16]
 [21 48]]


### Calculating recall, precision, and f1 scores

In [26]:
recall = recall_score(y_test, y_pred)
print(recall*100)

69.56521739130434


In [27]:
precision = precision_score(y_test, y_pred)
print(precision*100)

75.0


In [28]:
f1 = f1_score(y_test, y_pred)
print(f1*100)

72.18045112781954


### Building the interactive predictive picker function

In [29]:
passenger = {
    "age":30,
    "sex":"male",
    "pclass":3,
    "embarked":"S",
    "sibsp":0,
    "parch":1
}

family_size = passenger["sibsp"] + passenger["parch"] + 1
is_alone = family_size == 1

In [30]:
X_sample = pd.DataFrame([{
    "age":passenger["age"],
    "sex":passenger["sex"],
    "pclass":passenger["pclass"],
    "embarked":passenger["embarked"],
    "family_size": family_size,
    "is_alone": is_alone
}])

X_sample

,age,sex,pclass,embarked,family_size,is_alone
0,30,male,3,S,2,False


In [31]:
prediction = pipeline.predict(X_sample)
print(prediction)

[0]


In [32]:
probability = pipeline.predict_proba(X_sample)
print(probability*100)

[[89.57767459 10.42232541]]


In [33]:
def predict_survival(passenger_info):

    family_size = passenger_info["sibsp"] + passenger_info["parch"] + 1
    is_alone = family_size == 1

    X_passenger = pd.DataFrame([{
        "pclass": passenger_info["pclass"],
        "sex": passenger_info["sex"],
        "age": passenger_info["age"],
        "embarked": passenger_info["embarked"],
        "family_size": family_size,
        "is_alone": is_alone
    }])

    prediction = pipeline.predict(X_passenger)
    probability = pipeline.predict_proba(X_passenger)

    prediction = int(prediction[0])
    survival_probability = float(probability[0][1])

    result_text = "Survived" if prediction == 1 else "Didn't survive"
    survival_percentage = round(survival_probability * 100, 2)

    return survival_percentage, result_text


In [34]:
predict_survival(passenger)

(10.42, "Didn't survive")

### Exporting the model

In [35]:
jl.dump(pipeline, "/Users/samhithkonidena/My Data Journey/Data Job Prep/Portfolio/Analysis/Titanic_Analysis/Scripts/models/picker_pipeline.pkl")

['/Users/samhithkonidena/My Data Journey/Data Job Prep/Portfolio/Analysis/Titanic_Analysis/Scripts/models/picker_pipeline.pkl']